#  實務範例：訂單狀態查詢與授權檢查

### 商業情境

在 CRM Chatbot 的情境中，允許客戶查詢自己的訂單狀態，但不允許查詢其他客戶的訂單。
所以要在 Chatbot 中加入授權檢查的機制。

如果是自己的訂單，Chatbot 會查詢訂單狀態並回覆；
如果是其他客戶的訂單，Chatbot 會拒絕查詢並回覆「無法存取」。

因為要執行授權檢查，衍生的需求，需要知道目前登入的使用者的 `user_id`
如此，才能使用 `user_id` 與 `order_id` 查詢授權資料，判斷該使用者是否有權限查詢指定的訂單。

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# Custom AgentState

from typing import TypedDict
from langchain.agents import AgentState


class LastAuthCheck(TypedDict):
    order_id: str
    timestamp: str


# Custom AgentState
class OrderAgentState(AgentState):
    user_id: str
    last_auth_check: LastAuthCheck

In [9]:
# 授權檢查的工具函數

from langchain.tools import ToolRuntime, tool
from langgraph.types import Command
from datetime import datetime
from langchain.messages import ToolMessage


@tool
def check_order_auth(order_id: str, runtime: ToolRuntime) -> Command:
    """
    Check whether the current user is authorized to view an order.

    Args:
        user_id: The user ID of the current user.
        order_id: The order ID provided by the user.
        runtime: The ToolRuntime object injected by LangChain.
    """

    # 模擬的授權資料，實際系統應從資料庫或授權服務查詢
    order_authorizations = {
        "user_123": ["A1024", "B2048"],
        "user_456": ["C4096"],
    }

    # 從 Agent 的 custom state 讀取目前登入者
    user_id = runtime.state["user_id"]

    # 從獨立的授權表取得該 User 可以查看的訂單編號
    authorized_order_ids = order_authorizations.get(user_id, [])
    is_authorized = order_id in authorized_order_ids
    last_auth_check: LastAuthCheck = {
        "order_id": order_id,
        "timestamp": datetime.now().isoformat(),
    }

    return Command(
        # 更新 Agent state 中的 last_auth_check 欄位，記錄最後一次授權檢查的時間及檢查的 order_id
        update={
            "last_auth_check": last_auth_check,
            # 手動新增一個 ToolMessage 至 command 的 messages 欄位，記錄授權檢查結果
            "messages": [
                ToolMessage(
                    content=f"{{'order_id': '{order_id}', 'authorized': {is_authorized}}}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [10]:
# 查詢訂單狀態的工具函數
from lib.order_process_helper import get_order_status

In [11]:
# Update system prompt for the customer service assistant

# New rule
authorization_rules = """
Additional authorization rules:
- Before calling `get_order_status`, always call
  `check_order_authorization` with the same order ID.
- Only call `get_order_status` when `check_order_authorization` returns
  `authorized=True`.
- If authorization is denied, do not call `get_order_status`. Tell the user
  that the order cannot be accessed.
"""

# Update the system prompt to include the new authorization rules
from lib.order_process_helper import CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT

authorized_system_prompt = CUSTOMER_SERVICE_ASSISTANT_SYSTEM_PROMPT + "\n" + authorization_rules

In [12]:
# Create the agent

# Get the InMemorySaver checkpointer
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

# Create the tool list
tools = [check_order_auth, get_order_status]

# Create the agent with the updated system prompt and tools
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=authorized_system_prompt,
    checkpointer=checkpointer,
    state_schema=OrderAgentState
)



In [13]:
# Prepare the initial agent state and the thread id. 
from langchain.messages import HumanMessage

init_state = {
    "messages": [HumanMessage("請查詢訂單 A1024 的狀態。")],
    "user_id": "user_123"
}

thread_id = "order-support-1234"
runnable_config = {
    "configurable": {
        "thread_id": thread_id
    }
}

In [14]:
# Invoke the agent with the initial state and runnable config

resp_1 =  agent.invoke(
    input=init_state,
    config=runnable_config
)

In [27]:
from langchain_core.messages import AIMessage

def print_ai_messages_with_toolcalls(resp):
    """
    Print the tool calls for each AIMessage in the response.

    Args:
        resp: The response from the agent invocation.
    """
    ai_messages_with_toolcalls = [
        ai_message for ai_message in resp["messages"]
        if isinstance(ai_message, AIMessage) and hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0
    ]

    for index, ai_message in enumerate(ai_messages_with_toolcalls):
        print(f"\nAIMessage [{index}] tool calls:")
        from pprint import pp
        pp(ai_message.tool_calls)


# Print the tool calls for each AIMessage
print_ai_messages_with_toolcalls(resp_1)


AIMessage [0] tool calls:
[{'name': 'check_order_auth',
  'args': {'order_id': 'A1024'},
  'id': 'call_2z2p5KV3nB1myPCqUXZ0ttLY',
  'type': 'tool_call'}]

AIMessage [1] tool calls:
[{'name': 'get_order_status',
  'args': {'order_id': 'A1024'},
  'id': 'call_EfFvwBuixvA8G1P1QbQfgZu5',
  'type': 'tool_call'}]


In [26]:
# Second invocation with a different order ID to demonstrate authorization check
human_message_2 = HumanMessage("請查詢訂單 C4096 的狀態。")

response_2 = agent.invoke(
    input={
        "messages": [human_message_2]},
    config=runnable_config)

In [28]:
print_ai_messages_with_toolcalls(response_2)


AIMessage [0] tool calls:
[{'name': 'check_order_auth',
  'args': {'order_id': 'A1024'},
  'id': 'call_2z2p5KV3nB1myPCqUXZ0ttLY',
  'type': 'tool_call'}]

AIMessage [1] tool calls:
[{'name': 'get_order_status',
  'args': {'order_id': 'A1024'},
  'id': 'call_EfFvwBuixvA8G1P1QbQfgZu5',
  'type': 'tool_call'}]

AIMessage [2] tool calls:
[{'name': 'check_order_auth',
  'args': {'order_id': 'C4096'},
  'id': 'call_TxU7XKEuKieuWZJIVRIxn40P',
  'type': 'tool_call'}]


In [12]:
from pprint import pp
pp(response_2["messages"][-1].content)  # Print the last message content to see the authorization result

('抱歉，無法存取訂單 C4096 的狀態。系統顯示此訂單的存取未授權。\n'
 '\n'
 '請確認你已以該訂單所屬的帳戶登入，或聯繫客服提供相關證明以便協助查詢。如果你願意，我也可以在你解決授權後，再幫你重新查詢該訂單的狀態。')


In [28]:
# Check the last authorization check in the agent state
stateSnapshot = agent.get_state(config=runnable_config)
pp(stateSnapshot.values["last_auth_check"])  # Print the last authorization check details

{'order_id': 'C4096', 'timestamp': '2026-08-04T10:55:23.688058'}
